<h1 style="text-align: center; font-family: 'menlo'; color: #F2D2BD; font-size:50px;">
  <span style="background-color: #811331; padding: 5px 10px; border-radius: 5px; display: inline-block;">
     Tests images
  </span>
</h1>

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from matplotlib import rc #runtime configuration
rc('font', size=20)
rc('axes', grid=True)

import ccdproc as ccdp #automatiza o processo de redução de images CCD
from astropy.nddata import CCDData 
import astropy.units as u
from astropy.visualization import hist
from astropy.stats import mad_std #median absolute deviation

from convenience_functions import show_image
import image_sim as imsim

seed = os.getenv('GUIDE_RANDOM_SEED', None)
if seed is not None:
    seed = int(seed)
noise_rng = np.random.default_rng(seed)

In [ ]:
#parametros

gain = 1.0
bias_level = 1100
read_noise_electrons = 5
dark_current_e = 0.1  # e-/pix/s
dark_exposure = 100   # segundos
star_exposure = 30    # segundos
sky_counts = 20
max_stars_counts = 2000

image_shape = [1000, 1000]

#bias

n_bias = 10
bias_frames = []
for i in range(n_bias):
    bias_im = imsim.bias(np.zeros(image_shape), bias_level, realistic = True)
    noise_im = imsim.read_noise(np.zeros(image_shape), read_noise_electrons, gain = gain)
    bias_frames.append(bias_im + noise_im)

print(f'gerados {n_bias} bias frames')

#dark frames

n_dark = 10
dark_frames = []
for i in range (n_dark):
    bias_im = imsim.bias(np.zeros(image_shape), bias_level, realistic = True)
    noise_im = imsim.read_noise(np.zeros(image_shape), read_noise_electrons, gain=gain)
    dark_im = imsim.dark_current(np.zeros(image_shape), dark_current_e, dark_exposure, gain=gain, hot_pixels = True)
    dark_frames.append(bias_im + noise_im + dark_im)

print(f'gerados {n_dark} dark frames')

# flat frames --> flat = bias + noise + flat_sensitivity x (sky_bright)

n_flat = 10
flat_frames = []
flat_sensitivity = imsim.sensitivity_variations(np.zeros(image_shape))
for i in range (n_flat):
    sky_flat = imsim.sky_background(np.zeros(image_shape), 5000, gain=gain) #céu beeeem brilhante, 5000 = ADU
    bias_im = imsim.bias(np.zeros(image_shape), bias_level, realistic = True)
    noise_im = imsim.read_noise(np.zeros(image_shape), read_noise_electrons, gain=gain)
    flat_frames.append(bias_im + noise_im + flat_sensitivity * sky_flat)

print(f'gerados {n_flat} flat frames')

In [ ]:
#gerar a images -->  bias + noise + dark + flat x (sky + stars)

n_light = 3 #3 imagesss
light_frames = []
for i in range (n_light):
    bias_im = imsim.bias(np.zeros(image_shape), bias_level, realistic = True)
    dark_im = imsim.dark_current(np.zeros(image_shape), dark_current_e, star_exposure, gain=gain, hot_pixels = True)
    sky_im = imsim.sky_background(np.zeros(image_shape), sky_counts, gain=gain)
    stars_im = imsim.stars(np.zeros(image_shape), 50, max_counts=max_stars_counts)
    noise_im = imsim.read_noise(np.zeros(image_shape), read_noise_electrons, gain=gain) 
    light = bias_im + noise_im + flat_sensitivity * (sky_im + stars_im)
    light_frames.append(light)

print(f'gerados {n_light} light frames')

In [ ]:
#passa p CCDData

bias_ccd = [CCDData(b, unit='adu') for b in bias_frames]
dark_ccd = [CCDData(d, unit='adu') for d in dark_frames]
flat_ccd = [CCDData(f, unit='adu') for f in flat_frames]
light_ccd = [CCDData(l, unit='adu') for l in light_frames]

#adicionar metadados
for i, ccd in enumerate (dark_ccd):
    ccd.header['exptime'] = dark_exposure #exposição em segundos

for i, ccd in enumerate (light_ccd):
    ccd.header['exptime'] = star_exposure

for ccd in flat_ccd:
    ccd.header['exptime'] = 100
    
master_dark.header['exptime'] = 100

In [ ]:
#masters

master_bias = ccdp.combine(bias_ccd, method='median', sigma_clip=True, 
                           sigma_clip_low_thresh=5,
                          sigma_clip_high_thresh=5,
                          sigma_clip_func=np.ma.median)
print('Master bias prontoo')
show_image(master_bias.data, cmap='gray')
plt.title('Master bias (mediana de 10 frames)')

#dark eh com bias subtraído

dark_bias_subtracted = [ccdp.subtract_bias(d, master_bias) for d in dark_ccd]
master_dark = ccdp.combine(dark_bias_subtracted, method='median', sigma_clip=True,
                          sigma_clip_low_thresh=5,
                          sigma_clip_high_thresh=5)
print('Master dark prontoo')
show_image(master_dark.data, cmap='gray')
plt.title('Master dark (bias subtraído, mediana de 10 frames)')

#flat eh combinar flats com bias e dark subtraídos

flat_calibrated = []
for f in flat_ccd:
    f = ccdp.subtract_bias(f, master_bias)
    f = ccdp.subtract_dark(f, master_dark, exposure_time='exptime', exposure_unit=u.second, scale=True)
    flat_calibrated.append(f)
    
combined_flat = ccdp.combine(flat_calibrated, method='median', sigma_clip=True,
                            sigma_clip_low_thresh=5,
                            sigma_clip_high_thresh=5)
flat_norm_value = np.median(combined_flat.data)
master_flat = CCDData(combined_flat.data / flat_norm_value, unit='adu')
print('Master flat prontoo')
show_image(master_flat.data, cmap='gray')
plt.title('Master flat (normalizado)')

In [ ]:
#reduzir a imagem

raw_light = light_ccd[0]
print(f"antes da redução:")
print(f"  Min: {raw_light.data.min():.1f} ADU")
print(f"  Max: {raw_light.data.max():.1f} ADU")
print(f"  Média: {raw_light.data.mean():.1f} ADU")

#correções

reduced = ccdp.subtract_bias(raw_light, master_bias) #subtrair bias

#subtrair dark 
reduced = ccdp.subtract_dark(reduced, master_dark, exposure_time='exptime',
                             exposure_unit=u.second, scale=True)

#dividr pelo flat

reduced = ccdp.flat_correct(reduced, master_flat)

print(f"\nDepois da redução:")
print(f"  Min: {reduced.data.min():.1f} ADU")
print(f"  Max: {reduced.data.max():.1f} ADU")
print(f"  Média: {reduced.data.mean():.1f} ADU")

In [ ]:
#comparar

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

#imagens brutas
im1 = axes[0, 0].imshow(raw_light.data, cmap='gray', 
                         vmin=np.percentile(raw_light.data, 1),
                         vmax=np.percentile(raw_light.data, 99.9))
axes[0, 0].set_title('RAW')
axes[0, 0].axis('off')

#bias que foi subtraído
im2 = axes[0, 1].imshow(master_bias.data, cmap='gray',
                         vmin=np.percentile(master_bias.data, 1),
                         vmax=np.percentile(master_bias.data, 99))
axes[0, 1].set_title('Master Bias')
axes[0, 1].axis('off')

#dark que foi subtraído
im3 = axes[0, 2].imshow(master_dark.data, cmap='gray',
                         vmin=np.percentile(master_dark.data, 1),
                         vmax=np.percentile(master_dark.data, 99))
axes[0, 2].set_title('Master Dark')
axes[0, 2].axis('off')

#resultados

im4 = axes[1, 0].imshow(reduced.data, cmap='gray',
                         vmin=np.percentile(reduced.data, 1),
                         vmax=np.percentile(reduced.data, 99.9))
axes[1, 0].set_title('Reduced (bias - dark - flat)')
axes[1, 0].axis('off')

#flat
im5 = axes[1, 1].imshow(master_flat.data, cmap='gray',
                         vmin=0.94, vmax=1.06)  # flat fica perto de 1.0
axes[1, 1].set_title('Master Flat (normalized)')
axes[1, 1].axis('off')

# diferença
diff = raw_light.data - reduced.data
im6 = axes[1, 2].imshow(diff, cmap='hot',
                         vmin=np.percentile(diff, 1),
                         vmax=np.percentile(diff, 99))
axes[1, 2].set_title('Removed')
axes[1, 2].axis('off')

# Colorbars
for ax, im in zip(axes.flat, [im1, im2, im3, im4, im5, im6]):
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
#comparar com histograma

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

#imagem raw
hist(raw_light.data.flatten(), bins='freedman', ax=axes[0], 
     label='Raw', alpha=0.6, color='red')
axes[0].set_title('Distribuição de pixels - RAW')
axes[0].set_xlabel('Counts (ADU)')
axes[0].set_ylabel('Número de pixels')
axes[0].legend()

#imagem reduzida
hist(reduced.data.flatten(), bins='freedman', ax=axes[1],
     label='Reduzida', alpha=0.6, color='green')
axes[1].set_title('Distribuição de pixels - REDUZIDA')
axes[1].set_xlabel('Counts (ADU)')
axes[1].set_ylabel('Número de pixels')
axes[1].legend()

plt.tight_layout()
plt.show()

<h1 style="text-align: left; font-family: 'menlo'; color: #811331; font-size:35px;">
  <span style="background-color: #F2D2BD; padding: 5px 10px; border-radius: 5px; display: inline-block;">
     Links
  </span>
</h1>

https://learn.astropy.org/tutorials/FITS-images.html<br>https://docs.astropy.org/en/stable/nddata/ccddata.html<br>https://ccdproc.readthedocs.io/en/latest/ccddata.html<br>https://www.astropy.org/ccd-reduction-and-photometry-guide/v/dev/notebooks/00-00-Preface.html<br>https://github.com/astropy/ccd-reduction-and-photometry-guide<br>